# Powiększanie zbioru treningowego: HerBERT [MASK] + GPT-2

Klasyfikacja `ton`. Zbiór walidacyjny i testowy są takie same jak w `bert_peft.ipynb`, augmentujemy tylko zbiór treningowy. Na końcu trenujemy HerBERT + LoRA na zbiorze powiększonym i porównujemy z modelem trenowanym bez augmentacji.

In [1]:
import random

import pandas as pd
import torch
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import classification_report, f1_score
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, pipeline

from lm_common import *

random.seed(SEED)
torch.manual_seed(SEED)
train_df, val_df, test_df = load_data()
train_df.ton.value_counts()

/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ton
zwykły       344
agresywny    136
Name: count, dtype: int64

## 4. Augmentacja tokenem [MASK] (HerBERT)

W każdym tekście maskujemy po kolei ok. 15% słów i każde zastępujemy losową propozycją z top-5 modelu MLM (pomijając oryginalne słowo).

In [2]:
fill = pipeline("fill-mask", model="allegro/herbert-base-cased", device=DEVICE)
MASK = fill.tokenizer.mask_token


def mask_augment(text, ratio=0.15, top_k=5):
    words = text.split()
    candidates = [i for i, w in enumerate(words) if w.isalpha() and len(w) > 2]
    for i in random.sample(candidates, max(1, int(len(candidates) * ratio))):
        masked = " ".join(words[:i] + [MASK] + words[i + 1 :])
        options = [o["token_str"].strip() for o in fill(masked, top_k=top_k)]
        options = [o for o in options if o and o.lower() != words[i].lower() and o.isalpha()]
        if options:
            words[i] = random.choice(options)
    return " ".join(words)


for text in train_df.text[:3]:
    print("oryginał:", text)
    print("MLM:     ", mask_augment(text))
    print()

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 204/204 [00:00<00:00, 27199.38it/s]


[transformers] BertForMaskedLM LOAD REPORT from: allegro/herbert-base-cased
Key                             | Status     |  | 
--------------------------------+------------+--+-
cls.sso.sso_relationship.weight | UNEXPECTED |  | 
cls.sso.sso_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight        | UNEXPECTED |  | 
bert.pooler.dense.bias          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


oryginał: Również badania wskazują, że od 40 do 60% dzieci w wieku szkolnym i przedszkolnym ma zaburzenia mowy, a zatem wymagają one opieki logopedów. Mam szczerą wątpliwość, czy proponowana zmiana rzeczywiście rozwiąże w Polsce braki zawodów deficytowych. Chciałabym zapytać projektodawców, skąd ustalenie, że wymiar nieprzekraczający 9 godzin tygodniowo będzie uzasadniony dla zatrudnienia nauczycieli specjalistów w tych szkołach niepublicznych. Dziękuję bardzo.


/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:438.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/shinji/studia/nlp/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:174: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:498.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


MLM:      Również badania wskazują, że od 40 do 60% młodzieży w wieku dorosłym i przedszkolnym ma braki mowy, a zatem wymagają one zatrudnienia logopedów. Mam szczerą wątpliwość, czy proponowana zmiana rzeczywiście rozwiąże w Polsce braki zawodów deficytowych. Chciałabym zainteresować projektodawców, skąd ustalenie, że wymiar nieprzekraczający 9 godzin tygodniowo będzie uzasadniony dla zatrudnienia nauczycieli specjalistów w tych szkołach niepublicznych. Dziękuję bardzo.

oryginał: Szanowna Pani Marszałek! Panie Ministrze! Wysoka Izbo! Słowa mają ogromną moc. Czasem mogą podnosić na duchu, ale często potrafią ranić. I jest to ból gorszy niż ból fizyczny. Hejt, groźby, publiczne poniżanie to nie są tylko przykre słowa. To są również akty agresji, które mogą niszczyć i niszczą życie drugiego człowieka. Nasze prawo musi chronić wszystkich obywateli w równym stopniu, w szczególności osoby z niepełnosprawnościami, seniorów, kobiety czy osoby nieheteronormatywne. Nie możemy tolerować sytuacj

In [3]:
mlm_aug = pd.DataFrame([
    {"text": mask_augment(r.text), "label": r.label, "źródło": "mlm"}
    for r in train_df.itertuples()
    for _ in range(2 if r.ton == "agresywny" else 1)
])
len(mlm_aug)

616

## 5. Augmentacja generatywna (GPT-2)

Pierwsze słowa tekstu treningowego służą jako prompt, a GPT-2 pisze dalszy ciąg. Nowy tekst dziedziczy etykietę tekstu źródłowego. Najpierw porównujemy różne hiperparametry generacji.

In [4]:
del fill
torch.cuda.empty_cache()

GPT = "sdadas/polish-gpt2-medium"
gpt_tok = AutoTokenizer.from_pretrained(GPT)
gpt = AutoModelForCausalLM.from_pretrained(GPT, torch_dtype=torch.float16).to(DEVICE).eval()


@torch.no_grad()
def generate(texts, prompt_words=12, n=1, max_new_tokens=70, **params):
    prompts = [" ".join(t.split()[:prompt_words]) for t in texts]
    out = []
    for p in prompts:
        enc = gpt_tok(p, return_tensors="pt").to(DEVICE)
        ids = gpt.generate(**enc, do_sample=True, max_new_tokens=max_new_tokens, num_return_sequences=n, no_repeat_ngram_size=3, pad_token_id=gpt_tok.eos_token_id, **params)
        out.append([gpt_tok.decode(i, skip_special_tokens=True).strip() for i in ids])
    return out


SETTINGS = {
    "greedy-ish: T=0.3": dict(temperature=0.3),
    "T=0.7, top_k=50": dict(temperature=0.7, top_k=50),
    "T=1.0, top_p=0.9": dict(temperature=1.0, top_p=0.9, top_k=0),
    "T=1.0, top_k=50, top_p=0.95": dict(temperature=1.0, top_k=50, top_p=0.95),
    "T=1.4, top_k=0, top_p=1.0": dict(temperature=1.4, top_k=0, top_p=1.0),
}
examples = train_df[train_df.ton == "agresywny"].text[:2].tolist()
for name, params in SETTINGS.items():
    print(f"=== {name}")
    for text, (gen,) in zip(examples, generate(examples, **params)):
        print("-", gen)
    print()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading weights:  94%|█████████▍| 274/292 [00:00<00:00, 2736.02it/s]

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 2467.24it/s]


[transformers] GPT2LMHeadModel LOAD REPORT from: sdadas/polish-gpt2-medium
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== greedy-ish: T=0.3


- Między sobą się spierali, posłowie PiS-u, posłowie Solidarnej Polski, kto gdzie więcej w regionie zarobi, a kto mniej. W końcu się dogadali. - To jest bardzo dobra wiadomość. Nie tylko dla regionu, ale i dla Polski - mówi poseł PiS-owski. - W tym roku w regionie mamy rekordową liczbę nowych inwestycji. To jest też dobra wiadomość dla mieszkańców. To oznacza, że w regionie jest więcej miejsc pracy. To
- Dlaczego Niemcy nie weszli do tego programu? Zastanawialiście się? Bo są głupi, nie wiedzą, że w Polsce nie ma żadnego programu. A jak nie ma, to nie ma też pieniędzy. A poza tym, to co z tego, że nie ma pieniędzy? Przecież to nie jest program dla Polski. To jest program, który ma na celu tylko i wyłącznie to, żeby Niemcy nie mieli pieniędzy. I to jest program na lata.

=== T=0.7, top_k=50


- Między sobą się spierali, posłowie PiS-u, posłowie Solidarnej Polski, kto gdzie więcej pieniędzy dostanie. I to się stało. Dziś mamy taką sytuację. I nie jest to kwestia ani PiS-owska, ani PiSowska. To jest kwestia PO-owska. W związku z tym, panie pośle, jest to pytanie, jak to się skończy. Czy PiS przejmie władzę po wyborach? Czy PO będzie rządzić do końca kadencji? Czy może będzie rządzić
- Dlaczego Niemcy nie weszli do tego programu? Zastanawialiście się? Bo są głupi, ale w dupie to mają, bo przecież wiedzą, że to tylko na chwilę, a potem mogą się cieszyć życiem.

=== T=1.0, top_p=0.9


- Między sobą się spierali, posłowie PiS-u, posłowie Solidarnej Polski, kto gdzie więcej namieszał, w sejmiku, a potem w komisji sprawiedliwości i petycji, może była kampania wyborcza, potem zostało to załatwione – zaznaczył. – Rozumiem, że na tym etapie po wyborach stanowisko będzie takie, że nie będziemy się tym zajmować, ale jeśli będzie możliwość, to zajmiemy się tym – stwierdził. Dopytywany, czy partia miała więcej do powiedzenia
- Dlaczego Niemcy nie weszli do tego programu? Zastanawialiście się? Bo są głupi, bo wpadli na taki właśnie genialny pomysł! Poza tym jeśli Polak może dać się oszukać niemiecko-żydowskiemu systemowi to ja i ty możemy. Cała nasza przeszłość jest zbudowana na tym by nas wykorzystać. Wyrżnąć kogo się da, zrujnować. Najlepiej pod butem Hitlera, by na nasze śmieci nie trafiło nawet 10% wartości. No i my

=== T=1.0, top_k=50, top_p=0.95


- Między sobą się spierali, posłowie PiS-u, posłowie Solidarnej Polski, kto gdzie więcej zarobi). W lutym 2018 r. w rozmowie z portalem Onet.pl poseł Marek Suski stwierdził, że w PiS dominuje przekonanie, iż celem PiS-owskiego rządu jest zniszczenie trójpodziału władzy. Jego słowa cytowały " Gazeta Wyborcza" i Radio Zet. - To był po prostu atak na PiS, czyli na ludzi. A co
- Dlaczego Niemcy nie weszli do tego programu? Zastanawialiście się? Bo są głupi, czy co? Nie wiem, nie znam się, ale się wypowiem. To, co tu napisałeś jest prawdą i nie można być naiwnym. Po drugie to są "niebezpieczni ludzie", to bardzo złe słowo, na to się po prostu nie nadaje, bo jest to określenie pejoratywne, używane przez ludzi pozbawionych podstawowej wiedzy. Nie każdy,

=== T=1.4, top_k=0, top_p=1.0


- Między sobą się spierali, posłowie PiS-u, posłowie Solidarnej Polski, kto gdzie więcej wirdoli odczyże, kto wbije gwoździe w bloku głosowań. Masz kultury ekspert jeno ekspercie autoiwno do reszty wpojony szorować gary xD Już wtedy stwierdziłem, komu wolno, więc jego wspólnego lodówki zamknął wszelki nie wstając od wysokich robót muszą lecieć wszystkie gotować, płótno zajmu mecenas Olbrychski zaostrzył zwyczajem zaryby wszyscy tracą formę
- Dlaczego Niemcy nie weszli do tego programu? Zastanawialiście się? Bo są głupi, znalazł sobie człowieczka z Rzymu który będzie zarabiał jak zrobił prezydenta... no soft). Tak brać rodzine w pole. No świetnie... Seno bieda ludzie... taquito trupa jak pokazują zamawiali i wystawiczy 9zesztylet prezydentka jakims laptopem korzysta z Google Maxx = itp itd problemy z wzięciem nowych de tych budoscmniejszych



Niska temperatura daje powtarzalne, bezpieczne zdania, a T=1.4 bez obcinania rozkładu daje tekst w dużej mierze niespójny. Do augmentacji bierzemy kompromis: T=0.9, top_k=50, top_p=0.95. Generujemy dwa warianty dla tekstów agresywnych (klasa mniejszościowa) i jeden dla zwykłych.

In [5]:
GEN = dict(temperature=0.9, top_k=50, top_p=0.95)
gen_rows = []
for r in train_df.itertuples():
    n = 2 if r.ton == "agresywny" else 1
    for g in generate([r.text], n=n, **GEN)[0]:
        gen_rows.append({"text": g, "label": r.label, "źródło": "gpt2"})
gpt_aug = pd.DataFrame(gen_rows)
del gpt
torch.cuda.empty_cache()
gpt_aug.sample(5, random_state=SEED)

,text,label,źródło
512,"1. W jaki sposób ZUS i KRUS dotrą do osób, któ...",0,gpt2
403,Szanowna Pani Marszałek! Panie Ministrze! Wyso...,0,gpt2
534,Szanowny Panie Marszałku! Panie i Panowie Posł...,1,gpt2
293,Wielce Szanowna Pani Marszałkini! Szanowne Oso...,1,gpt2
549,"Jestem dumny, że zdajemy dziś celująco niełatw...",1,gpt2


In [6]:
aug_df = pd.concat([train_df.assign(źródło="oryginał")[["text", "label", "źródło"]], mlm_aug, gpt_aug], ignore_index=True)
pd.crosstab(aug_df.źródło, aug_df.label.map(dict(enumerate(TON))))

label,agresywny,zwykły
źródło,,
gpt2,272,344
mlm,272,344
oryginał,136,344


## Trening: bez augmentacji vs z augmentacją

In [7]:
MODEL = "allegro/herbert-base-cased"
tok = AutoTokenizer.from_pretrained(MODEL)


def fit(df, name):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.1, target_modules=["query", "key", "value"]))
    train(model, tok, seq_dataset(df, tok), seq_dataset(val_df, tok), epochs=10, name=name)
    return predict(model, tok, test_df.text.tolist())


results = {}
for name, df in {"bez augmentacji": train_df, "MLM + GPT-2": aug_df}.items():
    preds = fit(df, f"aug-{name}")
    print(name)
    print(classification_report(test_df.label, preds, target_names=TON, digits=3))
    results[name] = {
        "accuracy": (preds == test_df.label).mean(),
        "macro F1": f1_score(test_df.label, preds, average="macro"),
        "F1 agresywny": f1_score(test_df.label, preds),
        "n treningowych": len(df),
    }
pd.DataFrame(results).T.round(3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35044.99it/s]


[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those para

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

Map: 100%|██████████| 480/480 [00:00<00:00, 21663.16 examples/s]

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

... (9 outputs left out)

Map: 100%|██████████| 60/60 [00:00<00:00, 13568.68 examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.588012,0.293373,0.900000,0.872159
2,0.343397,0.266017,0.866667,0.822222
3,0.265315,0.296311,0.883333,0.840849
4,0.187001,0.282661,0.916667,0.895507
5,0.130346,0.471196,0.850000,0.784946
6,0.101729,0.431275,0.883333,0.832736
7,0.061450,0.562361,0.866667,0.803601
8,0.053992,0.456916,0.866667,0.822222
9,0.032802,0.531230,0.883333,0.832736
10,0.035743,0.577826,0.883333,0.832736


MLM + GPT-2
              precision    recall  f1-score   support

      zwykły      0.900     0.837     0.867        43
   agresywny      0.650     0.765     0.703        17

    accuracy                          0.817        60
   macro avg      0.775     0.801     0.785        60
weighted avg      0.829     0.817     0.821        60



,accuracy,macro F1,F1 agresywny,n treningowych
bez augmentacji,0.800,0.744,0.625,480.0
MLM + GPT-2,0.817,0.785,0.703,1712.0
